# Unlearning Evaluation

## What This Is
Unlearning is only useful if you can verify it worked. Evaluation addresses the question: *did the model actually forget the requested data?*

## Verification Approaches

1. **Membership Inference Attack (MIA)**: Apply a membership inference attack targeting the forget set. After successful unlearning, the attack should perform at chance (AUC ≈ 0.5).
2. **Forget set accuracy**: On the original task, the model should perform at chance on the forget set if it truly forgot the labels.
3. **Weight distance from retraining**: Compare the unlearned model's weights to the gold-standard retrained model.
4. **Activation similarity**: Compare intermediate activations between the unlearned model and the retrained model.
5. **Forget set loss distribution**: The loss distribution on the forget set should match a never-seen test set, not the training set.

## The Verification Challenge
Approximate unlearning methods can "appear" to unlearn by simple masking tricks without actually changing model behavior. Robust evaluation requires multiple independent verification methods.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

np.random.seed(42)

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def train_logistic_np(X, y, lr=0.1, epochs=200):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    for _ in range(epochs):
        z = X @ w + b
        err = sigmoid(z) - y
        w -= lr * (X.T @ err) / n
        b -= lr * err.mean()
    return w, b

# Dataset setup
N, D = 600, 10
X = np.random.randn(N, D)
true_w = np.random.randn(D)
y = (sigmoid(X @ true_w) > 0.5).astype(float)

# Splits
forget_idx = list(range(100, 130))  # 30 samples
retain_idx = [i for i in range(500) if i not in forget_idx]
test_idx = list(range(500, 600))   # 100 test samples

X_forget, y_forget = X[forget_idx], y[forget_idx]
X_retain, y_retain = X[retain_idx], y[retain_idx]
X_test, y_test = X[test_idx], y[test_idx]

# Train models
w_orig, b_orig = train_logistic_np(X[list(range(500))], y[list(range(500))])
w_retrain, b_retrain = train_logistic_np(X_retain, y_retain)

# Simple gradient ascent unlearning
def gradient_ascent_unlearn(w_init, b_init, X_f, y_f, X_r, y_r, lr=0.03, epochs=30):
    w, b = w_init.copy(), b_init
    for _ in range(epochs):
        dw_f = (X_f.T @ (sigmoid(X_f @ w + b) - y_f)) / len(y_f)
        dw_r = (X_r.T @ (sigmoid(X_r @ w + b) - y_r)) / len(y_r)
        w = w + 0.05 * dw_f - lr * dw_r
    return w, b

w_unlearn, b_unlearn = gradient_ascent_unlearn(
    w_orig, b_orig, X_forget, y_forget, X_retain, y_retain
)
print('Models ready for evaluation.')


In [ ]:
# Evaluation 1: Loss-based Membership Inference
def compute_losses(X, y, w, b):
    z = X @ w + b
    probs = np.clip(sigmoid(z), 1e-10, 1 - 1e-10)
    return -(y * np.log(probs) + (1 - y) * np.log(1 - probs))

def mia_auc(forget_losses, test_losses):
    """AUC for distinguishing forget set from test set. Should be ~0.5 after unlearning."""
    all_losses = np.concatenate([forget_losses, test_losses])
    # Members (forget) are labeled 1; non-members (test) labeled 0
    labels = np.array([1]*len(forget_losses) + [0]*len(test_losses))
    # Higher loss = more likely non-member in simple threshold attack
    # Flip sign: lower loss = more member-like
    return roc_auc_score(labels, -all_losses)

models = [
    ('Original (no unlearning)', w_orig, b_orig),
    ('Retrained (gold standard)', w_retrain, b_retrain),
    ('Gradient ascent unlearned', w_unlearn, b_unlearn),
]

print('Unlearning Evaluation Report')
print('=' * 70)
print(f'{'Model':<30} {'Forget Acc':>10} {'Test Acc':>9} {'MIA AUC':>8} {'W Dist':>8}')
print('-' * 70)

def acc(X, y, w, b):
    return ((sigmoid(X @ w + b) > 0.5).astype(int) == y).mean()

for name, w, b in models:
    f_acc = acc(X_forget, y_forget, w, b)
    t_acc = acc(X_test, y_test, w, b)
    f_losses = compute_losses(X_forget, y_forget, w, b)
    t_losses = compute_losses(X_test, y_test, w, b)
    mia = mia_auc(f_losses, t_losses)
    wdist = np.linalg.norm(w - w_retrain)
    print(f'{name:<30} {f_acc:>10.3f} {t_acc:>9.3f} {mia:>8.3f} {wdist:>8.4f}')

print('\nInterpretation:')
print('  Forget Acc:  should approach 0.5 (random chance) after unlearning')
print('  MIA AUC:     should approach 0.5 after unlearning')
print('  W Dist:      distance from gold-standard retrained model — lower = better')


In [ ]:
# Evaluation 2: Loss distribution comparison
# After unlearning, the forget set loss distribution should
# match the test (non-member) distribution

import statistics

print('Loss Distribution Analysis')
print('=' * 60)

for name, w, b in models:
    f_losses = compute_losses(X_forget, y_forget, w, b)
    r_losses = compute_losses(X_retain, y_retain, w, b)
    t_losses = compute_losses(X_test, y_test, w, b)
    
    print(f'\n{name}:')
    print(f'  Retain loss:  mean={r_losses.mean():.4f} std={r_losses.std():.4f}')
    print(f'  Forget loss:  mean={f_losses.mean():.4f} std={f_losses.std():.4f}')
    print(f'  Test loss:    mean={t_losses.mean():.4f} std={t_losses.std():.4f}')
    
    # KS-style comparison: is forget distribution closer to test or retain?
    diff_to_retain = abs(f_losses.mean() - r_losses.mean())
    diff_to_test = abs(f_losses.mean() - t_losses.mean())
    verdict = 'test (unlearned)' if diff_to_test < diff_to_retain else 'retain (still memorized)'
    print(f'  Forget set behaves more like: {verdict}')

print('\nAfter successful unlearning:')
print('  Forget loss distribution should match test (non-member) distribution')
